# Energy calibration of the fluorescence detector

Measures the channel → energy relation `E[eV] = gain × channel + offset` of the
SDD (XIA `xia_det0`) on a reference sample, with `xrf_calibration`:

1. **coarse match** — the calibrant's tabulated lines are matched to the measured
   peaks over a range of gains and offsets around the starting guess (10 eV/channel);
2. **global fit** — the constrained line model of `xrf_fit` with gain and offset free:
   every line sits at its tabulated energy with tabulated relative intensities, so
   unresolved doublets (Kα1/Kα2, Lβ1–4, …) are placed correctly. The detector
   resolution is fitted at the same time;
3. **line checks** — the strongest peaks are refitted one by one with only an energy
   shift free, to test the linearity of the response;
4. **intensity check** — measured atom ratios against the certified loadings (a test
   of the excitation and detection-efficiency models, not of the calibration).

Reference sample: AXO Dresden RF-type, C10 loading — a few atomic layers each of Pb,
La, Pd, Mo, Cu, Fe on a 200 nm Si membrane. Other calibrants can be added to
`xrf_calibration.CALIBRANTS`.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from bliss_hdf5_reader import BlissScan
from xrf_beamline import BM32_WHITE_BEAM
from xrf_calibration import CALIBRANTS, calibrate, plot_calibration
from xrf_fit import Calibration, Detector, element_line_groups

## 1. Settings

In [ ]:
# ---------------------------------------------------------------- measurement
H5_FILE = Path("/data/visitor/blc16918/bm32/20260303/RAW_DATA/XRF_calib_C10/"
               "XRF_calib_C10_surface/XRF_calib_C10_surface.h5")
SCANS = [9]                  # one or more scans of the calibrant (ct, loopscan, ...); spectra are summed
MCA = "xia_det0"

# ---------------------------------------------------------------- calibrant
CALIBRANT = CALIBRANTS["AXO RF C10"]
EXTRA_ELEMENTS = ["Ar", "Ca"]  # Ar: air between sample and detector; Ca: contamination seen on this sample
ROI_EV = (2500.0, 19500.0)     # fitted energy window (Pd K, 21 keV, is beyond the last channel)

# ---------------------------------------------------------------- fit
START_GAIN, START_OFFSET = 10.0, 0.0   # eV/channel, eV
BKG_ORDER = 5                          # polynomial background over the ROI (scattered white beam)

# Detector: starting resolution (fitted) and detection efficiency (fixed), as in the
# analysis notebooks: KETEK VITUS H7 CH, 450 um Si, 1 um C window, ~20 cm of air.
DETECTOR0 = Detector.from_fwhm(fwhm_ev=135.0, at_energy_ev=5900.0, material="Si", fano=0.12,
                               gamma_ev=5.0, sensor_thickness_um=450.0, window_material="C",
                               window_thickness_um=1.0, air_path_cm=20.0)

# Excitation: BM32 white beam. It sets the relative line intensities within each
# element, and the intensity check.
E_GRID_KEV = np.arange(3.0, 40.0 + 1e-9, 0.010)
PHI = BM32_WHITE_BEAM(air_path_m=3.3).spectrum(E_GRID_KEV)[1]

OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

## 2. Spectrum

In [ ]:
spectra = []
for n in SCANS:
    scan = BlissScan(H5_FILE, n)
    print(scan.summary().splitlines()[0])
    data = np.asarray(scan.get(MCA), dtype=float)
    spectra.append(data.reshape(-1, data.shape[-1]).sum(axis=0))   # sum the points of the scan
y = np.sum(spectra, axis=0)
n_channels = y.size
print(f"\n{n_channels} channels, {y.sum():.3e} counts in total")

In [ ]:
# The spectrum with the calibrant lines placed with the starting guess.
start = Calibration(gain=START_GAIN, offset=START_OFFSET)
groups = element_line_groups(CALIBRANT.elements + EXTRA_ELEMENTS, *ROI_EV, families=("K", "L", "M"),
                             min_rel=0.1, incident_spectrum=(E_GRID_KEV, PHI), detector=DETECTOR0)
ch = np.arange(n_channels)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.semilogy(start.energy(ch) / 1000, np.maximum(y, 0.5), "k-", lw=0.7)
cmap = plt.get_cmap("tab10")
for i, g in enumerate(groups):
    for l in g.lines:
        ax.axvline(l.energy_ev / 1000, color=cmap(i % 10), lw=0.8, alpha=0.4 + 0.6 * l.rel,
                   label=g.label if l is g.reference else None)
ax.set_xlabel(f"Energy (keV), starting guess {START_GAIN:g} eV/channel")
ax.set_ylabel("Counts")
ax.set_xlim(0, start.energy(n_channels) / 1000)
ax.legend(ncol=5, fontsize=8)
ax.set_title(f"{H5_FILE.stem}, scan(s) {SCANS}: {CALIBRANT.name} lines (relative intensity >= 10 %)")
fig.tight_layout()

## 3. Calibration

In [ ]:
result = calibrate(y, CALIBRANT, DETECTOR0, roi_ev=ROI_EV, gain=START_GAIN, offset=START_OFFSET,
                   extra_elements=EXTRA_ELEMENTS, incident_spectrum=(E_GRID_KEV, PHI),
                   background_order=BKG_ORDER, escape=True)
print(result.summary())

In [ ]:
fig_fit, fig_checks = plot_calibration(result)
stem = f"{H5_FILE.stem}_scan{'-'.join(map(str, SCANS))}"
fig_fit.savefig(OUT_DIR / f"{stem}_energy_calibration_fit.png", dpi=150, bbox_inches="tight")
if fig_checks is not None:
    fig_checks.savefig(OUT_DIR / f"{stem}_energy_calibration_lines.png", dpi=150, bbox_inches="tight")

**Reading the line checks.** Each point is one peak refitted on its own: the energy
the calibration gives it minus its tabulated energy. For a linear detector response
they scatter around zero within their error bars (χ² vs 0 ≈ 1); a smooth trend with
energy means a non-linear response. Isolated outliers usually point to something
unmodelled next to that peak (another element, a sum peak) rather than to the
calibration. A few-eV offset between K and L lines is also common: satellite lines
and asymmetric line shapes move the measured centroids slightly away from the
tabulated diagram-line energies. The spread of the well-measured points is a fair
estimate of the real accuracy of the calibration, which is usually larger than its
statistical error.

**Intensity check.** The measured atom ratios depend on the white-beam spectrum and
the detection efficiency, so large deviations from the certificate say that those
models need work. They do not affect the calibration.

## 4. Comparison with the calibration in use

In [ ]:
current = Calibration()            # default of xrf_fit, used by the analysis notebooks
new = result.calibration
print(f"in use : E = {current.gain:.5f} x ch {current.offset:+.2f} eV")
print(f"new    : E = {new.gain:.5f} x ch {new.offset:+.2f} eV")
print(f"\n{'energy':>8} {'channel':>9} {'new - in use':>13} {'+/-':>6}")
for e in (3000, 5000, 8000, 10000, 15000, 20000):
    c = current.channel(e)
    print(f"{e/1000:6.1f} keV {c:9.1f} {new.energy(c) - e:+11.1f} eV {result.energy_error(c):6.1f}")

In [ ]:
# Fitted resolution, for the Detector of the analysis notebooks
d = result.detector
print(f"fitted: noise {d.noise_ev:.1f} eV, Fano {d.fano:.3f}, gamma {d.gamma_ev:.1f} eV")
print(f"{'energy':>8} {'FWHM fitted':>12} {'FWHM start':>11}")
for e in (3000, 5900, 8000, 10000, 15000):
    print(f"{e/1000:6.1f} keV {d.fwhm_ev(e):9.0f} eV {DETECTOR0.fwhm_ev(e):8.0f} eV")

## 5. Save

In [ ]:
record = {
    "file": str(H5_FILE), "scans": SCANS, "mca": MCA,
    "measured": BlissScan(H5_FILE, SCANS[0]).start_time,
    "roi_ev": list(ROI_EV), "extra_elements": EXTRA_ELEMENTS, "background_order": BKG_ORDER,
    **result.to_dict(),
}
path = OUT_DIR / f"{stem}_energy_calibration.json"
path.write_text(json.dumps(record, indent=2))
print(f"saved {path}\n")
print("For the analysis notebooks:")
print(f"CALIB = Calibration(gain={new.gain:.5f}, offset={new.offset:.2f})   # {CALIBRANT.name}, {record['measured'][:10]}")